In [0]:
import yaml

In [0]:
from pyspark.sql.functions import col,trim
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

In [0]:
config_path = '/Volumes/metadata__etl/datasets/config/datasets.yml'

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

display(config)

In [0]:
catalog = config['catalog']
schema = config['schema']

spark.sql(f'create catalog if not exists {catalog}')
spark.sql(f'use catalog {catalog}')

spark.sql(f'create schema if not exists {schema}')

In [0]:
for data in config["datasets"]:
    print(f"Processing {data['name']}")

    df = (
        spark.read
             .format(data["format"])
             .option("header", data["header"])
             .option("inferSchema", data["inferSchema"])
             .load(data["source"].replace('/Volumes/metadata_etl/default/datasets', '/Volumes/metadata__etl/datasets'))
    )

    if data["transformations"]["trim"]:
        for field in df.schema.fields:
            if isinstance(field.dataType, StringType):
                df = df.withColumn(field.name, trim(col(field.name)))

    if data["transformations"]["remove_duplicates"]:
        df = df.dropDuplicates()

    if data["transformations"]["null_check"]:
        df = df.na.drop()

    df.write \
      .mode("overwrite") \
      .format("delta") \
      .saveAsTable(f"{catalog}.{schema}.{data['target_table']}")

    print(f"{data['target_table']} loaded successfully")

In [0]:

for data in config["datasets"]:
    try:
        print(f"Processing {data['name']}")

        df = (
            spark.read
                 .format(data["format"])
                 .option("header", data["header"])
                 .option("inferSchema", data["inferSchema"])
                 .load(data["source"].replace('/Volumes/metadata_etl/default/datasets', '/Volumes/metadata__etl/datasets'))
        )

        
        if data["transformations"]["trim"]:
            for field in df.schema.fields:
                if isinstance(field.dataType, StringType):
                    df = df.withColumn(field.name, trim(col(field.name)))

        
        if data["transformations"]["remove_duplicates"]:
            df = df.dropDuplicates()

        
        if data["transformations"]["null_check"]:
            df = df.na.drop()

        (
            df.write
              .mode("overwrite")
              .format("delta")
              .saveAsTable(f"{catalog}.{schema}.{data['target_table']}")
        )

        print(f"✓ {data['target_table']} loaded successfully.")

    except Exception as e:
        print(f"✗ Failed to process {data['name']}: {e}")

In [0]:
%sql
show tables in metedata_etl.bronze


In [0]:
%sql
select * from metadata_etl.bronze.sales limit 10;
select * from metadata_etl.bronze.customer limit 10;
select * from metadata_etl.bronze.product limit 10;
select * from metadata_etl.bronze.region limit 10;
select * from metadata_etl.bronze.city limit 10;
select * from metadata_etl.bronze.vendors limit 10;
select * from metadata_etl.bronze.payments limit 10;
select * from metadata_etl.bronze.shipments limit 10;